In [1]:
import os, json, shutil, zipfile, glob

# ============================================================
# CONFIGURE: set this to where your adapter dataset is mounted
# Kaggle mounts datasets at: /kaggle/input/<dataset-slug>/
# e.g. if you named it "nemotron-v7-adapter", look inside that folder
# ============================================================
ADAPTER_ZIP_PATH = None   # auto-detect below (or set manually)

# Auto-detect adapter.zip from any attached input dataset
for candidate in sorted(glob.glob("/kaggle/input/models/manish756/nvidia-adapter/transformers/adapter_recreating_baseline_model/22")):
    ADAPTER_ZIP_PATH = candidate
    break

if ADAPTER_ZIP_PATH is None:
    # Also check if the files were uploaded unzipped
    for candidate in sorted(glob.glob("/kaggle/input/models/manish756/nvidia-adapter/transformers/adapter_recreating_baseline_model/22/adapter_model.safetensors")):
        ADAPTER_ZIP_PATH = os.path.dirname(candidate)  # directory, not zip
        print(f"Found unzipped adapter at: {ADAPTER_ZIP_PATH}")
        break

if ADAPTER_ZIP_PATH is None:
    raise FileNotFoundError(
        "Could not find adapter.zip in any /kaggle/input/* directory.\n"
        "1. Download adapter.zip from your training notebook output\n"
        "2. Upload it as a Kaggle dataset\n"
        "3. Add that dataset as input to this notebook"
    )

print(f"Adapter source: {ADAPTER_ZIP_PATH}")

Adapter source: /kaggle/input/models/manish756/nvidia-adapter/transformers/adapter_recreating_baseline_model/22


In [2]:
# ============================================================
# EXTRACT adapter files to /kaggle/working/
# ============================================================
WORK_DIR = "/kaggle/working"

REQUIRED = {"adapter_config.json", "adapter_model.safetensors"}

if ADAPTER_ZIP_PATH.endswith(".zip"):
    # Extract from zip
    with zipfile.ZipFile(ADAPTER_ZIP_PATH, "r") as zf:
        for name in zf.namelist():
            if os.path.basename(name) in REQUIRED:
                # Extract flat — no subdirectory
                data = zf.read(name)
                dest = os.path.join(WORK_DIR, os.path.basename(name))
                with open(dest, "wb") as f:
                    f.write(data)
                size_mb = len(data) / 1024 / 1024
                print(f"  Extracted: {os.path.basename(name)}  ({size_mb:.1f} MB)")
else:
    # Already a directory (unzipped upload)
    for fname in REQUIRED:
        src = os.path.join(ADAPTER_ZIP_PATH, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(WORK_DIR, fname))
            size_mb = os.path.getsize(src) / 1024 / 1024
            print(f"  Copied: {fname}  ({size_mb:.1f} MB)")
        else:
            print(f"  MISSING: {fname}")

# Verify both essential files are present
present = {f for f in REQUIRED if os.path.exists(os.path.join(WORK_DIR, f))}
missing = REQUIRED - present
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print(f"\nAll required adapter files present in {WORK_DIR}")

  Copied: adapter_config.json  (0.0 MB)
  Copied: adapter_model.safetensors  (3373.4 MB)

All required adapter files present in /kaggle/working


In [3]:
# ============================================================
# VERIFY adapter_config.json — sanity check before packaging
# ============================================================
config_path = os.path.join(WORK_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)

print("adapter_config.json contents:")
print(json.dumps(cfg, indent=2))

# Critical checks
targets = cfg.get("target_modules", [])
checks = [
    ("base_model = metric/...",   cfg.get("base_model_name_or_path") == "metric/nemotron-3-nano-30b-a3b-bf16"),
    ("rank = 32",                  cfg.get("r") == 32),
    ("alpha = 64",                 cfg.get("lora_alpha") == 64),
    ("out_proj IN targets",        "out_proj" in targets),
    ("gate_proj NOT in targets",   "gate_proj" not in targets),
]

print("\nVerification:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed: all_ok = False
    print(f"  [{status}] {name}")

if not all_ok:
    print("\nWARNING: Some checks failed — review before submitting!")

adapter_config.json contents:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.0,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "peft_version": "0.18.1",
  "qalora_group_size": 16,
  "r": 32,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "q_proj",
    "v_proj",
    "up_proj",
    "down_proj",
    "out_proj",
    "in_proj",
    "o_proj",
    "k_proj"
  ],
  "tar

In [4]:
# ============================================================
# PACKAGE submission.zip — exactly 2 files the competition expects
# ============================================================
# The competition's evaluation metric looks for these 2 files
# inside submission.zip in /kaggle/working:
#   - adapter_config.json
#   - adapter_model.safetensors

SUB_ZIP = os.path.join(WORK_DIR, "submission.zip")

if os.path.exists(SUB_ZIP):
    os.remove(SUB_ZIP)

with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(REQUIRED):
        fpath = os.path.join(WORK_DIR, fname)
        zf.write(fpath, arcname=fname)   # flat — no subdirectory inside zip

# Verify
with zipfile.ZipFile(SUB_ZIP) as zf:
    contents = zf.namelist()

sub_mb = os.path.getsize(SUB_ZIP) / 1024 / 1024

print(f"{'='*55}")
print(f"  SUBMISSION READY")
print(f"{'='*55}")
print(f"  File     : {SUB_ZIP}")
print(f"  Size     : {sub_mb:.1f} MB")
print(f"  Contents : {contents}")
assert set(contents) == REQUIRED, f"Wrong files: {contents}"

  SUBMISSION READY
  File     : /kaggle/working/submission.zip
  Size     : 765.7 MB
  Contents : ['adapter_config.json', 'adapter_model.safetensors']
